# Swara — worker Colab

Notebook ini menjalankan mesin suara Swara **di Colab kamu sendiri**, bukan di server TemanSeller. Audio referensi dan hasilnya tidak pernah melewati server kami — browser di `temanseller.com/akun/swara` memanggil notebook ini langsung.

**Sebelum mulai:** pastikan kamu sudah membeli lisensi Swara dan login di temanseller.com dengan akun yang sama.

**Cara pakai — tinggal jalankan, tidak perlu menulis atau mengubah kode apa pun:**
1. Menu **Runtime → Change runtime type** → pilih **T4 GPU**.
2. Jalankan semua sel dari atas ke bawah (▶ di tiap sel, atau Runtime → Run all).
3. Tunggu sampai sel terakhir mencetak sebuah link.
4. Buka link itu (atau salin lalu tempel di halaman `temanseller.com/akun/swara`).
5. Biarkan tab notebook ini tetap terbuka selama kamu memakai Swara. Menjalankan ulang notebook membuat sesi baru dan mengambil alih dari sesi lama.

**Jujur soal batasannya:** Google Colab gratis bisa memutus sesi kapan saja dan GPU tidak selalu tersedia. Kalau link berhenti bekerja, jalankan ulang dari sel pertama dan tempel link barunya.


## 1. Pasang paket yang dibutuhkan

In [ ]:
!pip -q install fastapi uvicorn pynacl requests python-multipart omnivoice


## 2. Siapkan mesin suara
Sesuaikan baris di bawah bila kamu memakai model/engine lain hasil ujimu sendiri.


In [ ]:
import torch
print('GPU terdeteksi:', torch.cuda.is_available(), '-', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '-')


## 3. Server & pemeriksaan lisensi
**Jangan diubah** — bagian ini yang memeriksa lisensi Swara-mu ke temanseller.com sebelum mau menghasilkan audio apa pun. Kodenya dipersingkat (bukan rahasia dagang, hanya supaya tidak gampang dibongkar sambil lalu) — pelindung yang sebenarnya ada di server TemanSeller (token bertanda tangan, berumur 15 menit, satu sesi aktif per lisensi), jadi walau baris-baris ini dibaca orang, tanpa token sah dari server, endpoint di bawah tetap menolak.

In [ ]:
import base64 as _b,json as _j,secrets as _s,time as _t
import requests as _rq
from fastapi import FastAPI,File,Form,HTTPException as _H,Request,UploadFile
from fastapi.responses import Response as _R
from nacl.exceptions import BadSignatureError as _Bse
from nacl.signing import VerifyKey as _Vk
_ab="https://temanseller.com"
_ao="https://temanseller.com"
SID=_s.token_hex(8)
_pk=None
def _gp():
    global _pk
    if _pk is None:
        _r=_rq.get(f"{_ab}/api/swara/pubkey",timeout=10);_r.raise_for_status()
        _pk=_Vk(_b.b64decode(_r.json()["pubkey"]))
    return _pk
def _vt(_z):
    if not _z or not _z.startswith("Bearer "):raise _H(401,"x1")
    _w=_z.removeprefix("Bearer ").strip()
    try:
        _p1,_p2=_w.split(".",1);_pd=_b.b64decode(_p1);_sg=_b.b64decode(_p2)
    except Exception:raise _H(401,"x2")
    try:_gp().verify(_pd,_sg)
    except _Bse:
        global _pk;_pk=None
        try:_gp().verify(_pd,_sg)
        except _Bse:raise _H(401,"x3")
    _c=_j.loads(_pd)
    if _c.get("exp",0)<_t.time():raise _H(401,"x4")
    if _c.get("sid")!=SID:raise _H(401,"x5")
    return _c
def run_tts(text,ref_audio_path):
    from omnivoice import OmniVoice
    _m=run_tts.__dict__.setdefault("_m",OmniVoice.from_pretrained("k2-fsa/OmniVoice"))
    return _m.generate(text=text,ref_audio=ref_audio_path)
app=FastAPI()
@app.middleware("http")
async def _mw(request:Request,call_next):
    _o=request.headers.get("origin")
    _rs=_R(status_code=204) if request.method=="OPTIONS" else await call_next(request)
    if _o==_ao:
        _rs.headers["Access-Control-Allow-Origin"]=_ao
        _rs.headers["Access-Control-Allow-Headers"]="authorization,content-type"
        _rs.headers["Access-Control-Allow-Methods"]="POST,GET,OPTIONS"
    return _rs
@app.get("/health")
def _hc():return {"ok":True,"sid":SID}
@app.post("/generate")
async def _gn(request:Request,ref:UploadFile=File(...),text:str=Form(...)):
    _vt(request.headers.get("authorization"))
    if not text.strip():raise _H(422,"x6")
    import tempfile,os
    with tempfile.NamedTemporaryFile(suffix=".audio",delete=False) as _f:
        _f.write(await ref.read());_rp=_f.name
    try:_wv=run_tts(text.strip()[:2000],_rp)
    finally:os.unlink(_rp)
    return _R(content=_wv,media_type="audio/wav")
def _ot(port):
    import os,re,subprocess
    if not os.path.exists("./cloudflared"):
        subprocess.run(["wget","-q","https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64","-O","cloudflared"],check=True)
        subprocess.run(["chmod","+x","cloudflared"],check=True)
    _pr=subprocess.Popen(["./cloudflared","tunnel","--url",f"http://localhost:{port}"],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
    _dl=_t.time()+30
    while _t.time()<_dl:
        _ln=_pr.stdout.readline()
        _mt=re.search(r"https://[-\w]+\.trycloudflare\.com",_ln)
        if _mt:return _mt.group(0)
    raise RuntimeError("x7")
def start(port=8000):
    import threading,uvicorn
    threading.Thread(target=lambda:uvicorn.run(app,host="0.0.0.0",port=port,log_level="warning"),daemon=True).start()
    _t.sleep(2)
    _pu=_ot(port)
    _u=_b.b64encode(_pu.encode()).decode()
    _lk=f"{_ab}/akun/swara#sid={SID}&u={_u}"
    print("Notebook siap. Tempel link ini di halaman Swara (temanseller.com/akun/swara):\n");print(_lk)
    return _lk


## 4. Jalankan
Sel ini mencetak link untuk ditempel di `temanseller.com/akun/swara`.

In [ ]:
link = start()
